# Ollama on Colab — Proof of Concept

Runs a local LLM inside a free Colab runtime, with no HPC dependency.

**Run all cells top to bottom.** Total setup is about 2 minutes on a GPU runtime.

If you want a GPU: *Runtime -> Change runtime type -> T4 GPU*. The notebook works
without one — it just picks a smaller model and runs slower.

## 1. Install Ollama and start the server

In [11]:
!sudo dnf install zstd  pciutils  lshw -y


No read/execute access in current directory, moving to /
Last metadata expiration check: 0:16:15 ago on Sun 09 Aug 2026 02:00:32 PM EDT.
Package zstd-1.4.4-1.el8.x86_64 is already installed.
Package pciutils-3.7.0-3.el8.x86_64 is already installed.
Package lshw-B.02.19.2-6.el8_10.1.x86_64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!


In [12]:
pip install torch numpy


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
# Ollama's install script. ~30s.
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> Enabling and starting ollama service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [14]:
import subprocess, time, requests

# Ollama is a server. Our instance has no init system, so background it ourselves
# and hold the handle for the life of the runtime.
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Poll until it answers instead of sleeping a fixed amount.
for attempt in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=1).raise_for_status()
        print(f"Ollama is up (took ~{attempt}s)")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Re-run this cell.")

Ollama is up (took ~0s)


## 2. Pick a model that fits this runtime

The point of this cell is that **nobody gets stuck**. It checks what hardware
Colab actually handed you and picks a model that will run on it. Every cell
after this one is identical either way.

In [15]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    MODEL = "qwen3:8b" if vram_gb >= 12 else "qwen3:4b"
    print(f"GPU: {gpu} ({vram_gb:.1f} GB)")
else:
    MODEL = "qwen3:1.7b"
    print("No GPU assigned - falling back to CPU.")

print(f"Using model: {MODEL}")

No GPU assigned - falling back to CPU.
Using model: qwen3:1.7b


In [16]:
# Pull the weights. Ollama's CDN is fast on Colab (~100 MB/s), so even the
# 8B (5.2 GB) lands in about a minute.
#
# NOTE: do NOT point OLLAMA_MODELS at Google Drive. Drive is a FUSE mount and
# llama.cpp mmaps these files - it is slower and less reliable than re-pulling.
# Drive is for your work (section 5), not for weights.
!ollama pull {MODEL};

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 3d0b790534fe: 100% ▕██████████████████▏ 1.4 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 517ccaff02fe: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


## 3. Smoke test

In [17]:
!pip install -q ollama

import ollama

response = ollama.chat(
    model=MODEL,
        messages=[{"role": "user", "content": "In one paragraph: what is the origin of Golden Retrievers?"}],
        think=False,  # Qwen3 is a hybrid reasoning model; off by default keeps demos snappy.
)

print(response["message"]["content"])


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Golden Retrievers originated in the United States in the early 1930s. They were developed by Dr. Edward L. W. "Bud" MacMillan, a veterinarian in the Midwest, who bred a mix of the American Fox Terrier and the German Shepherd to create a versatile working dog. The breed was named after the golden color of their coat, and they were initially bred for hunting, particularly in the forests of the United States. Today, Golden Retrievers are popular as family pets and working dogs, often used in search and rescue, therapy, and law enforcement.


## 4. Context engineering — the part that actually matters

Same model, same question. The only thing that changes is the context:
a role, an explicit task, and a required output format.

This is the 1:30-2:00 block made concrete.

In [18]:
import json
import textwrap

SYSTEM = textwrap.dedent("""
    You are a research assistant helping triage academic abstracts.
    You are precise and you never speculate beyond the text you are given.
""").strip()

SCHEMA = {
    "type": "object",
    "properties": {
        "method": {"type": "string", "description": "The primary method used"},
        "domain": {"type": "string"},
        "is_empirical": {"type": "boolean"},
        "sample_size": {"type": ["integer", "null"]},
    },
    "required": ["method", "domain", "is_empirical", "sample_size"],
}

ABSTRACT = (
    "We surveyed 1,204 undergraduate students across three institutions to measure "
    "the relationship between sleep duration and working-memory performance. "
    "Participants completed a seven-day actigraphy protocol followed by n-back testing."
)

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Extract structured metadata from this abstract:\n\n{ABSTRACT}"},
    ],
    format=SCHEMA,   # Ollama constrains decoding to this schema.
    think=False,
    options={"temperature": 0},  # Extraction is not a creative task.
)

record = json.loads(response["message"]["content"])
print(json.dumps(record, indent=2))

{
  "method": "survey",
  "domain": "education",
  "is_empirical": true,
  "sample_size": 1204
}


### Why this cell is the whole workshop in miniature

- **Role** in the system prompt narrows the model's behavior.
- **`format=SCHEMA`** makes the output machine-readable, so it can feed the next
  step instead of being read by a human. This is what turns a chat into a pipeline.
- **`temperature=0`** makes it repeatable — run it twice, get the same record.

A repeatable task with structured output is the unit your harness is built from.

## 5. Make it a repeatable task, and persist the results

Now the same operation over a batch, written somewhere that survives the runtime.

In [19]:
def extract(abstract: str) -> dict:
    """One repeatable research task: abstract in, structured record out."""
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Extract structured metadata from this abstract:\n\n{abstract}"},
        ],
        format=SCHEMA,
        think=False,
        options={"temperature": 0},
    )
    return json.loads(response["message"]["content"])


CORPUS = [
    ABSTRACT,
    "This paper develops a variational framework for turbulence closure in "
    "magnetohydrodynamic flows. No experimental data are analyzed.",
    "We interviewed 32 clinicians about barriers to adopting electronic health "
    "records, using a semi-structured protocol and thematic analysis.",
]

records = [extract(a) for a in CORPUS]

for r in records:
    print(json.dumps(r))

{"method": "survey", "domain": "education", "is_empirical": true, "sample_size": 1204}
{"method": "variational framework", "domain": "magnetohydrodynamic flows", "is_empirical": false, "sample_size": 0}
{"method": "qualitative", "domain": "healthcare", "is_empirical": true, "sample_size": 32}


In [20]:
# Persist to Drive. 

## What this proves

1. A local LLM runs on free Colab with no HPC involvement.
2. The runtime tier is handled automatically — nobody is blocked on GPU allocation.
3. Structured output makes a model call into a pipeline step.
4. Drive gives persistence for work products, with weights staying on local disk.

Everything above transfers to Ollama on HPC unchanged except the setup cells —
`ollama.chat()` against a local server is the same call either way.